In [2]:
# Install necessary libraries for the project
!pip install unsloth trl peft accelerate bitsandbytes # Install libraries using pip command

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 11.6 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
import torch # Import the torch library

# Check if CUDA is available and print the device name
print(f"CUDA Available {torch.cuda.is_available()}") # Print whether CUDA is available
print(f"Device Name {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}") # Print the name of the CUDA device if available, otherwise print 'None'

CUDA Available True
Device Name Tesla T4


In [6]:
# Define a list of data pairs, each containing a natural language query and its corresponding pandas code
data_pairs = [{"nl":"Create a column 'cum_sum' with cumulative sum per user sorted by timestamp.","code":"df['cum_sum'] = df.sort_values('timestamp').groupby('user')['value'].cumsum()"}, {"nl":"Select only numeric columns and standardize them to mean 0, std 1.","code":"from sklearn.preprocessing import StandardScaler\nnum_cols = df.select_dtypes('number').columns\ndf[num_cols] = StandardScaler().fit_transform(df[num_cols])"}, {"nl":"Compute time since first purchase per customer.","code":"df['first_purchase'] = df.groupby('customer')['date'].transform('min')\ndf['days_since_first'] = (df['date'] - df['first_purchase']).dt.days"}, {"nl":"Create pivot table showing average revenue per product per region.","code":"pivot = df.pivot_table(index='region', columns='product', values='revenue', aggfunc='mean')"}, {"nl":"Filter rows where 'score' is in the top 10% per group.","code":"df['top10pct'] = df.groupby('group')['score'].transform(lambda x: x >= x.quantile(0.9))"}, {"nl":"Forward fill missing values within each group sorted by date.","code":"df = df.sort_values('date').groupby('category').ffill()"}, {"nl":"Convert string column with multiple delimiters into lists.","code":"df['tags_list'] = df['tags'].str.split('[,;|]')"}, {"nl":"Explode multiple list columns into separate rows.","code":"for col in ['tags_list','categories']: df = df.explode(col)"}, {"nl":"Compute difference between consecutive events per user.","code":"df['diff'] = df.groupby('user')['value'].diff()"}, {"nl":"Detect consecutive duplicate rows and mark them.","code":"df['dup_seq'] = df.duplicated(subset=['user','event'], keep=False).astype(int)"}, {"nl":"Create a rolling max over 10 days for each stock.","code":"df['roll_max'] = df.groupby('stock')['price'].rolling(10, min_periods=1).max().reset_index(level=0, drop=True)"}, {"nl":"Compute z-score per group and flag outliers beyond 3 std.","code":"df['z'] = df.groupby('category')['value'].transform(lambda x: (x-x.mean())/x.std()); df['outlier'] = df['z'].abs()>3"}, {"nl":"Merge two DataFrames on multiple keys with outer join.","code":"merged = df1.merge(df2, on=['id','date'], how='outer')"}, {"nl":"Create a column indicating if a user ever made a purchase.","code":"df['ever_purchase'] = df.groupby('user')['purchase'].transform('max') > 0"}, {"nl":"Compute difference between a column and its group median.","code":"df['diff_median'] = df['value'] - df.groupby('group')['value'].transform('median')"}, {"nl":"Convert a wide format DataFrame to long format with multiple value_vars.","code":"long_df = df.melt(id_vars=['id','date'], value_vars=['v1','v2','v3'], var_name='metric', value_name='value')"}, {"nl":"Remove outliers using IQR method for each numeric column.","code":"for col in df.select_dtypes('number'):\n Q1 = df[col].quantile(0.25); Q3 = df[col].quantile(0.75); IQR = Q3-Q1\n df = df[~((df[col]<(Q1-1.5*IQR)) | (df[col]>(Q3+1.5*IQR)))]"}, {"nl":"Create a column with first non-null value per group.","code":"df['first_valid'] = df.groupby('user')['value'].transform('first')"}, {"nl":"Compute exponential moving average with span 10 per stock.","code":"df['ema10'] = df.groupby('stock')['price'].transform(lambda x: x.ewm(span=10, adjust=False).mean())"}, {"nl":"Merge multiple DataFrames stored in a list on a common column.","code":"from functools import reduce\ndf_merged = reduce(lambda left,right: left.merge(right, on='id', how='outer'), list_of_dfs)"},{"nl":"Add a column 'salary_net' subtracting tax 10% from 'salary'.","code":"df['salary_net'] = df['salary'] * 0.90"},
{"nl":"Create a column 'age_plus_ten' by adding 10 to 'age'.","code":"df['age_plus_ten'] = df['age'] + 10"},
{"nl":"Compute percent change of 'sales' month-over-month.","code":"df['sales_pct_change'] = df['sales'].pct_change()"},
{"nl":"Create a 30-day rolling sum of 'sales'.","code":"df['sales_30d_sum'] = df['sales'].rolling(window=30, min_periods=1).sum()"},
{"nl":"Compute exponentially weighted mean of 'price' with span=7.","code":"df['price_ewm'] = df['price'].ewm(span=7, adjust=False).mean()"},
{"nl":"Create a column 'prev_month_sales' using shift by 1.","code":"df['prev_month_sales'] = df['sales'].shift(1)"},
{"nl":"Calculate month-over-month growth using shift.","code":"df['mom_growth'] = (df['sales'] - df['sales'].shift(1)) / df['sales'].shift(1)"},
{"nl":"Rank salaries within each department.","code":"df['salary_rank_dept'] = df.groupby('department')['salary'].rank(method='dense', ascending=False)"},
{"nl":"Find the row-wise sum across columns 'q1','q2','q3'.","code":"df['total_q'] = df[['q1','q2','q3']].sum(axis=1)"},
{"nl":"Create a boolean column 'is_vip' if purchases > 10 and spend > 1000.","code":"df['is_vip'] = (df['purchases'] > 10) & (df['spend'] > 1000)"},
{"nl":"Use np.where to set 'status' to 'senior' if age>50 else 'junior'.","code":"import numpy as np\ndf['status'] = np.where(df['age']>50,'senior','junior')"},
{"nl":"Group by 'user' and compute cumulative count of events.","code":"df['event_cumcount'] = df.groupby('user').cumcount()+1"},
{"nl":"Compute group-wise normalized salary (z-score) per department.","code":"df['salary_z'] = df.groupby('department')['salary'].transform(lambda x:(x-x.mean())/x.std())"},
{"nl":"Filter rows where 'score' is between 70 and 90 inclusive.","code":"df_between = df[df['score'].between(70,90)]"},
{"nl":"Create a MultiIndex by ['region','store'] and sort it.","code":"df = df.set_index(['region','store']).sort_index()"},
{"nl":"Unstack the last level of the MultiIndex.","code":"df_unstack = df.unstack(level=-1)"},
{"nl":"Compute moving median of 'response_time' with window 5.","code":"df['resp_median_5'] = df['response_time'].rolling(5).median()"},
{"nl":"Fill missing 'rating' per user using forward fill within group.","code":"df['rating'] = df.groupby('user')['rating'].ffill()"},
{"nl":"Backfill missing 'end_date' column forward within each project.","code":"df['end_date'] = df.groupby('project')['end_date'].bfill()"},
{"nl":"Convert a column of JSON strings 'meta' into separate columns.","code":"from pandas import json_normalize\nmeta_df = json_normalize(df['meta'].apply(eval))\ndf = pd.concat([df.drop(columns=['meta']), meta_df], axis=1)"},
{"nl":"Explode a list-column 'tags' so each tag becomes a row.","code":"df = df.explode('tags').reset_index(drop=True)"},
{"nl":"Create lag and lead columns for 'value' by group 'id'.","code":"df['lag1'] = df.groupby('id')['value'].shift(1)\ndf['lead1'] = df.groupby('id')['value'].shift(-1)"},
{"nl":"Use merge_asof to align transactions to nearest previous price by time.","code":"merged = pd.merge_asof(transactions.sort_values('time'), prices.sort_values('time'), on='time', by='symbol', direction='backward')"},
{"nl":"Read a large CSV in chunks and compute sum of 'amount'.","code":"total=0\nfor chunk in pd.read_csv('large.csv', chunksize=100000):\n    total += chunk['amount'].sum()\nprint(total)"},
{"nl":"Write DataFrame to parquet using snappy compression.","code":"df.to_parquet('data.parquet', compression='snappy')"},
{"nl":"Read parquet into df and show dtypes.","code":"df = pd.read_parquet('data.parquet')\nprint(df.dtypes)"},
{"nl":"Convert categorical text column to 'category' dtype to save memory.","code":"df['category_col'] = df['category_col'].astype('category')"},
{"nl":"Factorize 'city' into integer codes.","code":"df['city_code'] = pd.factorize(df['city'])[0]"},
{"nl":"Use sample with weights based on 'popularity' column.","code":"sampled = df.sample(n=100, weights='popularity', random_state=1)"},
{"nl":"Compute rolling apply with a custom function on window=10.","code":"df['roll_custom'] = df['metric'].rolling(10).apply(lambda x: (x.mean()-x.median()))"},
{"nl":"Stack columns into long format then reset index.","code":"stacked = df.stack().reset_index(name='value')"},
{"nl":"Unpivot columns A,B,C into rows with id_vars ['id','date'].","code":"melted = df.melt(id_vars=['id','date'], value_vars=['A','B','C'], var_name='metric', value_name='val')"},
{"nl":"Create indicator columns for nulls for selected cols.","code":"for c in ['a','b','c']:\n    df[c+'_isnull'] = df[c].isnull()"} ,
{"nl":"Use transform to broadcast group sums to original rows.","code":"df['grp_sum'] = df.groupby('team')['score'].transform('sum')"},
{"nl":"Compute top 3 items per group using nlargest.","code":"top3 = df.groupby('group').apply(lambda x: x.nlargest(3,'score')).reset_index(drop=True)"},
{"nl":"Apply string extract to pull area code from phone number.","code":"df['area_code'] = df['phone'].str.extract(r'\\((\\d{3})\\)')"},
{"nl":"Create time series index and convert to Period monthly frequency.","code":"df['date'] = pd.to_datetime(df['date'])\ndf = df.set_index('date')\ndf.index = df.index.to_period('M')"},
{"nl":"Compute month end totals using to_timestamp after resample on period index.","code":"monthly = df.to_timestamp().resample('M')['sales'].sum()"},
{"nl":"Use shift and cumsum to convert daily changes into cumulative series.","code":"df['cum'] = df['change'].cumsum()"},
{"nl":"Use assign to create multiple new columns in a chain.","code":"df = df.assign(sales_k=lambda x: x['sales']/1000, profit_margin=lambda x: x['profit']/x['sales'])"},
{"nl":"Remove rows with duplicates across multiple columns ['a','b','c'].","code":"df = df.drop_duplicates(subset=['a','b','c'])"},
{"nl":"Normalize numeric columns using Min-Max scaling in place.","code":"num = df.select_dtypes(include='number').columns\ndf[num] = (df[num] - df[num].min())/(df[num].max()-df[num].min())"},
{"nl":"Use query with an external variable to filter rows.","code":"threshold=50\ndf_filtered = df.query('score > @threshold')"},
{"nl":"Create a rolling rank of 'score' within a 7-day window.","code":"df['rolling_rank'] = df['score'].rolling('7D').apply(lambda x: pd.Series(x).rank().iloc[-1])"},
{"nl":"Replace multiple regex patterns in a text column in one call.","code":"df['text'] = df['text'].replace({r'\\bfoo\\b':'bar', r'\\d+':'NUM'}, regex=True)"},
{"nl":"Compute correlation between two columns grouped by category using corr.","code":"corrs = df.groupby('cat').apply(lambda x: x['a'].corr(x['b'])).reset_index(name='corr')"},
{"nl":"Use explode on a column of dicts after converting to list of dicts.","code":"df = df.explode('attributes'); df = pd.concat([df.drop(columns=['attributes']), pd.json_normalize(df['attributes'])], axis=1)"},
{"nl":"Use map to replace codes with names from a dictionary.","code":"mapping = {'A':'Alpha','B':'Beta'}\ndf['name'] = df['code'].map(mapping)"},
{"nl":"Compute weighted average of 'score' by 'weight' per group.","code":"wa = df.groupby('group').apply(lambda g: (g['score']*g['weight']).sum()/g['weight'].sum()).reset_index(name='weighted_avg')"},
{"nl":"Use pivot_table with margins to get totals in pivot.","code":"pt = df.pivot_table(index='region', columns='product', values='sales', aggfunc='sum', margins=True)"},
{"nl":"Drop columns with more than 50% missing values.","code":"thresh = len(df)*0.5\ndf = df.dropna(axis=1, thresh=thresh)"},
{"nl":"Use df.filter to select columns that start with 'temp_'.","code":"temps = df.filter(regex='^temp_')"},
{"nl":"Save DataFrame to Excel with multiple sheets.","code":"with pd.ExcelWriter('out.xlsx') as w:\n    df.to_excel(w, sheet_name='data')\n    df.head(10).to_excel(w, sheet_name='sample')"} ,
{"nl":"Read HTML tables from a webpage and pick the first table.","code":"tables = pd.read_html('https://example.com')\ndf_table = tables[0]"},
{"nl":"Use update to modify df values with another DataFrame of same index.","code":"df.update(update_df)"},
{"nl":"Use combine_first to fill missing values from another DataFrame.","code":"df = df.combine_first(other_df)"} ,
{"nl":"Compute pairwise correlation matrix for selected numeric cols.","code":"corr = df[['a','b','c','d']].corr()"},
{"nl":"Use sample(frac=0.2) to take a 20% random subset reproducibly.","code":"subset = df.sample(frac=0.2, random_state=42)"},
{"nl":"Convert wide table to long with stack and name index levels.","code":"long = df.set_index(['id','date']).stack().reset_index(name='value')"},
{"nl":"Select columns by dtype: only floats.","code":"float_cols = df.select_dtypes(include=['float']).columns\nfloat_df = df[float_cols]"},
{"nl":"Rename index axis and columns axis names.","code":"df = df.rename_axis(index='user_id', columns='metrics')"},
{"nl":"Use pd.cut with quantile-based bins using qcut.","code":"df['quantile_bin'] = pd.qcut(df['score'], q=4, labels=False)"},
{"nl":"Find consecutive dates gaps per user using diff on sorted dates.","code":"df['date']=pd.to_datetime(df['date'])\ndf = df.sort_values(['user','date'])\ndf['gap_days']=df.groupby('user')['date'].diff().dt.days"},
{"nl":"Compute month over month sales growth grouped by store.","code":"df['sales_prev'] = df.groupby('store')['sales'].shift(1)\ndf['mom'] = (df['sales']-df['sales_prev'])/df['sales_prev']"},
{"nl":"Use merge with indicator to see join result status.","code":"merged = df.merge(other,on='id', how='outer', indicator=True) ; merged['_merge'].value_counts()"},
{"nl":"Change column order to put ['id','name'] first then others.","code":"cols = ['id','name'] + [c for c in df.columns if c not in cols]\ndf = df[cols]"},
{"nl":"Group by multiple cols and pivot the aggregated value.","code":"g = df.groupby(['region','product'])['sales'].sum().reset_index()\npivot = g.pivot(index='region', columns='product', values='sales')"},
{"nl":"Use rolling with min_periods to avoid NaNs at start.","code":"df['roll_min'] = df['val'].rolling(window=5, min_periods=1).mean()"},
{"nl":"Compute within-group rank with pct=True to get percent rank.","code":"df['pct_rank'] = df.groupby('class')['score'].rank(pct=True)"},
{"nl":"Create time-aware index and use asfreq to reindex to business day frequency.","code":"df['date']=pd.to_datetime(df['date']); df=df.set_index('date').asfreq('B')"},
{"nl":"Use tz_localize and tz_convert on a datetime index.","code":"df['time']=pd.to_datetime(df['time']).dt.tz_localize('UTC').dt.tz_convert('Asia/Kolkata')"},
{"nl":"Use to_datetime with dayfirst=True for ambiguous dates.","code":"df['date'] = pd.to_datetime(df['date'], dayfirst=True)"},
{"nl":"Use eval to create a new column 'net' as revenue - cost.","code":"df.eval('net = revenue - cost', inplace=True)"},
{"nl":"Use query to filter rows where colA>colB and flag them.","code":"df_flag = df.query('A > B')\ndf.loc[df['A']>df['B'],'flag']=True"},
{"nl":"Use where to replace values not meeting a condition with NaN.","code":"df['score'] = df['score'].where(df['score']>=0, other=pd.NA)"},
{"nl":"Compute covariance matrix for selected columns.","code":"cov = df[['x','y','z']].cov()"},
{"nl":"Use get_dummies with drop_first=True to avoid multicollinearity.","code":"dummies = pd.get_dummies(df['category'], drop_first=True); df = pd.concat([df, dummies], axis=1)"},
{"nl":"Compute the time difference in seconds between two timestamp columns.","code":"df['delta_s'] = (pd.to_datetime(df['t2']) - pd.to_datetime(df['t1'])).dt.total_seconds()"},
{"nl":"Use pivot_table with aggfunc as list to collect values per group.","code":"pt = df.pivot_table(index='user', values='item', aggfunc=list) ; pt = pt.reset_index()"},
{"nl":"Perform a cross join between two small DataFrames.","code":"a['key']=1; b['key']=1; cross = a.merge(b, on='key').drop('key',axis=1)"},
{"nl":"Use reindex to add missing dates and fill values forward.","code":"idx = pd.date_range(df['date'].min(), df['date'].max(), freq='D')\ndf = df.set_index('date').reindex(idx).ffill().rename_axis('date').reset_index()"},
{"nl":"Create a sparse DataFrame from many binary columns to save memory.","code":"sparse_df = df.astype(pd.SparseDtype('int', 0))"},
{"nl":"Use rename with a lambda to uppercase all column names.","code":"df = df.rename(columns=lambda x: x.upper())"},
{"nl":"Compute number of unique users per day using groupby + nunique.","code":"daily = df.groupby(pd.to_datetime(df['date']).dt.date)['user'].nunique().reset_index(name='unique_users')"},
{"nl":"Use str.contains with regex and na=False to safely filter.","code":"mask = df['text'].str.contains(r'error|fail', case=False, na=False)\nerrors = df[mask]"},
{"nl":"Aggregate with custom named columns using agg with tuples.","code":"agg = df.groupby('team').agg(total=('score','sum'), avg=('score','mean'), cnt=('score','count')).reset_index()"},
{"nl":"Use pd.concat to append multiple DataFrames with ignore_index True.","code":"df_all = pd.concat([df1, df2, df3], ignore_index=True)"},
{"nl":"Compute the exponential of numeric columns and store in new cols.","code":"num = df.select_dtypes(include='number').columns\nfor c in num:\n    df[c+'_exp'] = np.exp(df[c])"},
{"nl":"Detect outliers using IQR and flag them in a column.","code":"Q1 = df['val'].quantile(0.25); Q3 = df['val'].quantile(0.75); IQR=Q3-Q1\ndf['outlier'] = ((df['val'] < (Q1 - 1.5*IQR)) | (df['val'] > (Q3 + 1.5*IQR)))"},
{"nl":"Use sample with frac and replace=True to bootstrap rows.","code":"bootstrap = df.sample(frac=1, replace=True, random_state=0)"},
{"nl":"Use apply with axis=1 to create a new column combining two fields.","code":"df['fullname'] = df.apply(lambda r: f\"{r['first']} {r['last']}\", axis=1)"},
{"nl":"Use isin with a regex-like list to filter multiple variants of a city.","code":"cities=['New York','NYC','NewYork']\ndf[df['city'].isin(cities)]"},
{"nl":"Convert a numeric column with commas to int type.","code":"df['num']=df['num'].str.replace(',','').astype(int)"},
{"nl":"Use .astype with errors='ignore' to attempt conversion safely.","code":"df['maybe_int'] = pd.to_numeric(df['maybe_int'], errors='coerce').astype('Int64')"},
{"nl":"Use merge with suffixes to control overlapping column names.","code":"m = df.merge(other,on='id', how='left', suffixes=('_left','_right'))"},
{"nl":"Compute rolling apply groupwise by time using groupby and rolling.","code":"df['grp_roll_mean'] = df.groupby('id')['val'].rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)"},
{"nl":"Use .nlargest with subset to get top rows by multiple cols.","code":"top = df.nlargest(10, 'score')"},
{"nl":"Use index.isin to subset DataFrame by index labels.","code":"subset = df[df.index.isin(['id1','id2'])]"},
{"nl":"Use .interpolate(method='time') to fill time series gaps.","code":"df = df.sort_values('date'); df['value'] = df['value'].interpolate(method='time')"},
{"nl":"Compute difference between each value and the group's median.","code":"df['diff_median'] = df['val'] - df.groupby('group')['val'].transform('median')"},
{"nl":"Use .astype('timedelta64[D]') to convert timedelta to days.","code":"df['days'] = (pd.to_datetime(df['end']) - pd.to_datetime(df['start'])).astype('timedelta64[D]')"},
{"nl":"Use pd.to_numeric with downcast to reduce memory usage.","code":"df['small_int'] = pd.to_numeric(df['big_int'], downcast='integer')"},
{"nl":"Use .round on floats to 2 decimals in place.","code":"df['price'] = df['price'].round(2)"},
{"nl":"Use .duplicated keep=False to mark all duplicates.","code":"df['is_dup'] = df.duplicated(subset=['email','date'], keep=False)"},
{"nl":"Use freqstr in resample to get quarterly sums.","code":"quarterly = df.set_index('date').resample('Q')['revenue'].sum()"},
{"nl":"Convert categorical codes back to labels with inverse mapping.","code":"inv_map = {v:k for k,v in mapping.items()}\ndf['city_label'] = df['city_code'].map(inv_map)"},
{"nl":"Use .pipe to pass df to a custom function in a method chain.","code":"def clean(d):\n    d['x']=d['x'].fillna(0)\n    return d\n\ndf = df.pipe(clean).assign(z=lambda d: d['x']*2)"},
{"nl":"Use .mask to selectively replace values where condition is True.","code":"df['score'] = df['score'].mask(df['score'] < 0, 0)"},
{"nl":"Use .hist to quickly visualize distribution of a column.","code":"df['val'].hist(bins=50)"},
{"nl":"Use .corrwith to compute correlation between two DataFrames columns.","code":"corr_with = df[['a','b']].corrwith(other_df[['a','b']])"},
{"nl":"Use .query to filter rows using string methods via .str.","code":"df_filtered = df.query('name.str.startswith(\"A\")', engine='python')"},
{"nl":"Use replace with a dict to change multiple values across columns.","code":"df.replace({'status':{'0':'inactive','1':'active'}, 'type':{'x':'X'}}, inplace=True)"},
{"nl":"Use .to_sql to write a df to a SQL table using SQLAlchemy engine.","code":"from sqlalchemy import create_engine\nengine = create_engine('sqlite:///db.sqlite')\ndf.to_sql('table', engine, if_exists='replace', index=False)"},
{"nl":"Use read_sql_query to pull a subset from a database.","code":"import sqlite3\ncon=sqlite3.connect('db.sqlite')\ndf_db = pd.read_sql_query('SELECT id,name FROM table WHERE value>100', con)"},
{"nl":"Create a column with rolling correlation between two series with window 20.","code":"df['roll_corr'] = df['a'].rolling(20).corr(df['b'])"},
{"nl":"Use .clip to cap values between 0 and 1.","code":"df['prob'] = df['prob'].clip(lower=0, upper=1)"},
{"nl":"Use .astype('category').cat.codes to create numeric category codes.","code":"df['cat_code'] = df['cat'].astype('category').cat.codes"},
{"nl":"Use .to_pickle to save and load quickly.","code":"df.to_pickle('df.pkl')\ndf2 = pd.read_pickle('df.pkl')"},
{"nl":"Use .explode on a column of tuples and keep index.","code":"df = df.explode('coords')" },
{"nl":"Use .applymap to transform every cell with a function for small df.","code":"df = df.applymap(lambda x: x.strip() if isinstance(x,str) else x)"},
{"nl":"Compute rolling window std deviation with center=True.","code":"df['roll_std_center'] = df['val'].rolling(window=7, center=True).std()"},
{"nl":"Use .sample with replace True to simulate Monte Carlo draws.","code":"sims = df['returns'].sample(n=1000, replace=True).values"},
{"nl":"Use .groupby.transform('rank') to get rank per group without changing shape.","code":"df['rank_in_group'] = df.groupby('group')['score'].transform('rank')"},
{"nl":"Drop columns by position (e.g., drop second and fourth).","code":"cols = df.columns.tolist(); df = df.drop(columns=[cols[1], cols[3]])"},
{"nl":"Set multiple columns as index and then sort by index.","code":"df = df.set_index(['country','city']).sort_index()"},
{"nl":"Use .rename_axis to set multiindex names after stacking.","code":"s = df.stack(); s = s.rename_axis(['id','metric'])"},
{"nl":"Use .select_dtypes to drop object columns and keep numeric.","code":"num_df = df.select_dtypes(exclude=['object'])"},
{"nl":"Use .astype with dict to change multiple dtypes at once.","code":"df = df.astype({'id':'int64','flag':'bool'})"},
{"nl":"Use .to_csv with compression gzip to save disk space.","code":"df.to_csv('out.csv.gz', index=False, compression='gzip')"},
{"nl":"Use .groupby + apply with a custom function returning DataFrame (complex aggregation).","code":"def top_and_sum(g):\n    return pd.DataFrame({'top1':[g.nlargest(1,'score')['id'].iloc[0]], 'sum':[g['score'].sum()]})\nres = df.groupby('grp').apply(top_and_sum).reset_index()"},
{"nl":"Use .rolling with apply and raw=False to pass Series to function.","code":"df['custom'] = df['val'].rolling(5).apply(lambda s: s.rank().iloc[-1], raw=False)"},
{"nl":"Use .between_time on a datetime-indexed df to get rows between hours.","code":"df.index = pd.to_datetime(df['ts']); work = df.between_time('09:00','17:00')"},
{"nl":"Use .pct_change with periods=3 to compute quarter-over-quarter change.","code":"df['qtr_pct'] = df['value'].pct_change(periods=3)"},
{"nl":"Create a column that counts consecutive positive values per group.","code":"df['pos_run'] = (df['val']>0).groupby((df['val']>0).ne((df['val']<0).shift())).cumsum()"} ]

In [7]:
import json # Import the json library

# Open the file for writing the dataset in JSONL format
with open("pandas_finetune.jsonl","w",encoding='utf-8') as f: # Open the file in write mode with utf-8 encoding
  # Iterate through each data pair in the data_pairs list
  for pair in data_pairs: # Loop through each dictionary in data_pairs
    # Create a record dictionary in the desired format for the dataset
    record = { # Define the structure of a single record
        "messages":[ # List of messages for a conversation
            {"role":"system","content":"You are a helpful assistant that translates natural language into pandas code."}, # System message
            {"role":"user","content":pair["nl"]}, # User message with natural language query
            {"role":"assistant","content":pair["code"]} # Assistant message with corresponding pandas code
        ]
    }
    # Write the JSON record to the file, ensuring non-ASCII characters are handled
    f.write(json.dumps(record,ensure_ascii=False)+"\n") # Convert the record to a JSON string and write to the file, add newline
# Print a confirmation message
print("jsonl file created!") # Indicate that the file creation is complete

jsonl file created!


In [9]:
# Install the evaluate library
!pip install evaluate # Install the evaluate library using pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00


In [88]:
import json # Import the json library for working with JSON data
from datasets import load_dataset # Import load_dataset function from the datasets library for loading datasets
from transformers import ( # Import necessary classes from the transformers library
    AutoTokenizer, # AutoTokenizer for loading the tokenizer for a pre-trained model
    AutoModelForSeq2SeqLM, # AutoModelForSeq2SeqLM for loading a pre-trained sequence-to-sequence language model
    DataCollatorForSeq2Seq, # DataCollatorForSeq2Seq for preparing data batches for sequence-to-sequence models
    Seq2SeqTrainingArguments, # Seq2SeqTrainingArguments for defining training parameters for Seq2SeqTrainer
    Seq2SeqTrainer # Seq2SeqTrainer for fine-tuning sequence-to-sequence models
)
import evaluate # Import the evaluate library for computing evaluation metrics
from accelerate import Accelerator # Import Accelerator from accelerate for distributed training and mixed precision
from accelerate.state import AcceleratorState # Import AcceleratorState from accelerate.state for managing accelerator state

In [90]:
# Constants for the model, data, and output directory
MODEL_NAME = "Salesforce/codet5-small" # Name of the pre-trained model to use
DATA_PATH = "pandas_finetune.jsonl" # Path to the training data in JSONL format
OUTPUT_DIR = "./codet5-pandas" # Directory to save the fine-tuned model
MAX_IN = 128 # Maximum length for input sequences
MAX_OUT = 128 # Maximum length for output sequences

In [91]:
# Load the dataset from the JSONL file
dataset = load_dataset("json", data_files=DATA_PATH)["train"] # Load the dataset and select the 'train' split

# Split the dataset into training and evaluation sets
dataset = dataset.train_test_split(test_size=0.05, seed=42) # Split 5% of the data for testing with a fixed seed
train_ds, eval_ds = dataset["train"], dataset["test"] # Assign the splits to train_ds and eval_ds variables

In [92]:
# Load the tokenizer and model from the pre-trained model name
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True) # Load the tokenizer, using the fast version if available
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME) # Load the sequence-to-sequence model

In [93]:
# Preprocessing function to prepare the dataset for the model
# - We add a short prefix to inputs to make the task explicit (helps the model learn the task)
def preprocess(batch): # Define the preprocess function that takes a batch of data
    # Create input sequences by adding a prefix and extracting the natural language part
    inputs = ["translate to pandas: " + message[1]['content'] for message in batch["messages"]]
    # Tokenize the input sequences (for the encoder)
    model_inputs = tokenizer(inputs, max_length=MAX_IN, truncation=True, padding="max_length") # Tokenize with max length, truncation, and padding
    # Tokenize the target sequences (for the decoder) - use tokenizer.as_target_tokenizer for T5-style models
    with tokenizer.as_target_tokenizer(): # Context manager to set the tokenizer for target sequences
        labels = tokenizer([message[2]['content'] for message in batch["messages"]], max_length=MAX_OUT, truncation=True, padding="max_length") # Tokenize target sequences
    # Replace padding token ID in labels with -100 so the loss function ignores padding tokens
    model_inputs["labels"] = [ # Add the processed labels to the model inputs dictionary
        [(l if l != tokenizer.pad_token_id else -100) for l in label_ids] # Replace pad token ID with -100
        for label_ids in labels["input_ids"] # Iterate through label input IDs
    ]
    return model_inputs # Return the processed model inputs

# Apply the preprocessing function to the training and evaluation datasets
train_ds = dataset["train"].map(preprocess, batched=True, remove_columns=dataset["train"].column_names) # Apply preprocess to the training set in batches
eval_ds = dataset["test"].map(preprocess, batched=True, remove_columns=dataset["test"].column_names) # Apply preprocess to the evaluation set in batches

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

In [ ]:
# Initialize the BLEU metric for evaluation
bleu = evaluate.load("bleu") # Load the BLEU metric

# Define the compute_metrics function for evaluating the model's performance
def compute_metrics(eval_pred): # Define the function that takes evaluation predictions
    preds_ids, labels_ids = eval_pred # Unpack the prediction and label IDs

    # Move the input tensor to the same device as the model (GPU if available)
    input_tensor = torch.tensor(preds_ids).to(model.device) # Convert prediction IDs to a tensor and move to model's device

    # Generate tokens using the model based on the prediction IDs
    generated_tokens = model.generate( # Call the model's generate method
        input_ids=input_tensor, # Use the input tensor on the correct device
        max_length=MAX_OUT, # Set the maximum length of generated tokens
        num_beams=1 # Set the number of beams for beam search (1 means greedy decoding)
    )
    # preds are generated token ids (because predict_with_generate=True in training args)
    # Decode the generated token IDs back to text
    preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True) # Decode generated tokens, skipping special tokens
    # Decode the label token IDs back to text
    labels = tokenizer.batch_decode(labels_ids, skip_special_tokens=True) # Decode label tokens, skipping special tokens
    # Compute the BLEU score and return it
    return {"bleu": bleu.compute(predictions=preds, references=[[l] for l in labels])["bleu"]} # Compute BLEU and return the score

In [ ]:
# Initialize the Data Collator for sequence-to-sequence models
data_collator = DataCollatorForSeq2Seq( # Create an instance of DataCollatorForSeq2Seq
    tokenizer=tokenizer, # Pass the tokenizer to the data collator
    model=model, # Pass the model to the data collator
    padding=True, # Enable padding to the longest sequence in the batch
    label_pad_token_id=-100 # Set the label padding token ID to -100 (to be ignored by loss)
)

In [ ]:
# Configure the training arguments for the Seq2SeqTrainer
training_args = Seq2SeqTrainingArguments( # Create an instance of Seq2SeqTrainingArguments
    output_dir=OUTPUT_DIR, # Directory to save model checkpoints and outputs
    num_train_epochs=6, # Number of training epochs
    per_device_train_batch_size=8,    # Batch size per device during training (lower if OOM)
    per_device_eval_batch_size=8, # Batch size per device during evaluation
    predict_with_generate=True,      # Generate tokens during evaluation for metrics calculation
    eval_strategy="epoch", # Evaluate at the end of each epoch
    save_strategy="epoch", # Save model checkpoint at the end of each epoch
    logging_steps=200, # Log training progress every 200 steps
    learning_rate=5e-5, # Learning rate for the optimizer
    fp16=True,                        # Use mixed precision training (if GPU supports it)
    gradient_accumulation_steps=2,    # Number of updates steps to accumulate before performing a backward/update pass (increase if you lower batch_size)
    save_total_limit=2, # Limit the total number of saved checkpoints
    report_to="none", # Disable reporting to external services like Weights & Biases
    run_name="codet5-pandas-finetune", # Optional: specify a name for the training run
    # Removed max_new_tokens due to compatibility issues, might affect generation length during evaluation
)

In [ ]:
from accelerate import Accelerator # Import Accelerator for handling distributed training and mixed precision
from accelerate.state import AcceleratorState # Import AcceleratorState for managing accelerator state

# Reinitialize accelerator state (sometimes needed in interactive environments)
AcceleratorState._reset_state() # Reset the internal state of the accelerator
accelerator = Accelerator() # Create a new Accelerator instance

# Initialize the Seq2SeqTrainer
trainer = Seq2SeqTrainer( # Create an instance of Seq2SeqTrainer
    model=model, # Pass the loaded model
    args=training_args, # Pass the training arguments
    train_dataset=train_ds, # Pass the training dataset
    eval_dataset=eval_ds, # Pass the evaluation dataset
    tokenizer=tokenizer, # Pass the tokenizer (deprecated argument, processing_class is preferred in newer versions)
    data_collator=data_collator, # Pass the data collator
    compute_metrics=compute_metrics, # Pass the function to compute evaluation metrics
)

In [ ]:
# Start the training process
trainer.train() # Call the train method to start fine-tuning

# Save the fine-tuned model and tokenizer
trainer.save_model(OUTPUT_DIR) # Save the trained model to the specified output directory
tokenizer.save_pretrained(OUTPUT_DIR) # Save the tokenizer to the same directory

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM # Import necessary classes

# Load the fine-tuned tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("./codet5-pandas") # Load the tokenizer from the saved directory
model = AutoModelForSeq2SeqLM.from_pretrained("./codet5-pandas") # Load the model from the saved directory

# Create a text2text-generation pipeline
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=0) # Create a pipeline for text generation, specifying the model, tokenizer, and device (GPU 0)

In [95]:
# 6) Metric
bleu = evaluate.load("bleu") # Load the BLEU metric for evaluating generated text

def compute_metrics(eval_pred): # Define a function to compute evaluation metrics
    preds_ids, labels_ids = eval_pred # Unpack the prediction and label IDs from the evaluation prediction object

    # Debug: check for invalid pred token ids
    invalid_tokens = [] # Initialize an empty list to store invalid token IDs
    for batch in preds_ids: # Iterate through batches of predicted token IDs
        for tok in batch: # Iterate through each token ID in the batch
            if tok < 0 and tok != -100: # Check if the token ID is negative and not the special padding value (-100)
                invalid_tokens.append(tok) # Add invalid negative token IDs to the list
            if tok >= tokenizer.vocab_size: # Check if the token ID is greater than or equal to the tokenizer's vocabulary size
                invalid_tokens.append(tok) # Add out-of-vocabulary token IDs to the list
    if invalid_tokens: # If any invalid tokens were found
        print("Invalid tokens in predictions:", sorted(set(invalid_tokens))) # Print the unique invalid token IDs
        import sys # Import the sys module
        sys.exit("Stopping due to invalid token IDs in predictions") # Exit the program with an error message

    preds = tokenizer.batch_decode(preds_ids, skip_special_tokens=True) # Decode the predicted token IDs into human-readable strings, skipping special tokens
    labels = tokenizer.batch_decode(labels_ids, skip_special_tokens=True) # Decode the label token IDs into human-readable strings, skipping special tokens
    result = bleu.compute(predictions=preds, references=[[l] for l in labels]) # Compute the BLEU score between the predicted and label strings
    return {"bleu": result["bleu"]} # Return a dictionary containing the BLEU score

In [106]:
# 8) Train
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)



Step,Training Loss


('./codet5-pandas/tokenizer_config.json',
 './codet5-pandas/special_tokens_map.json',
 './codet5-pandas/vocab.json',
 './codet5-pandas/merges.txt',
 './codet5-pandas/added_tokens.json',
 './codet5-pandas/tokenizer.json')

## TESTING

In [107]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("./codet5-pandas")
model = AutoModelForSeq2SeqLM.from_pretrained("./codet5-pandas")
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=0)


Device set to use cuda:0


In [108]:
# Test the pipeline with a sample natural language query
out = pipe("translate to pandas: Sort dataframe by age column descending", max_length=128) # Use the pipeline to generate code for the query
print(out[0]["generated_text"]) # Print the generated code

Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


df['age'] = df['age'].sort_values()


In [111]:
# Test the pipeline with another sample query
out = pipe("translate to pandas:Group the data by department and calculate salary", max_length=128) # Use the pipeline to generate code
print(out[0]["generated_text"]) # Print the generated code

Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


df['salary'] = df.groupby('department')['salary'].sum()


In [112]:
# Test the pipeline with another sample query
out = pipe("translate to pandas:Count the number of entries per category in column type", max_length=128) # Use the pipeline to generate code
print(out[0]["generated_text"]) # Print the generated code

Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


df['count'] = df.count(category='category')


In [114]:
# Test the pipeline with another sample query
out = pipe("translate to pandas:Fill missing age values with the median age", max_length=128) # Use the pipeline to generate code
print(out[0]["generated_text"]) # Print the generated code

Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


df['age'] = df.fill_index('age')['median'].fill_index('age')['median'].fill_index('age').fill_index('age').fill_index('age').fill_index('age').fill_index('age').fill_index('age').fill_index('age').fill_index('age').fill_index('age').fill_index('age').fill_index('age').fill_index('age').fill


In [116]:
# Test the pipeline with another sample query
out = pipe("translate to pandas:Calculate the difference between score abd sum of scores", max_length=128) # Use the pipeline to generate code
print(out[0]["generated_text"]) # Print the generated code

Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


df['score'] = df['score'] - df['score'].sum()


In [ ]:
# This cell is empty and can be used for additional code or notes.